In [1]:
import polars as pl
import polars.selectors as cs
import os
import pyarrow.dataset as pads

In [3]:
####################################################
### Code Taken from SMT_Data_Starter
####################################################

# For the data_path argument, include the full file path to the folder that holds the data!    
def readDataSubset(table_type, data_path="/Users/jamesccoats/Documents/SMT-Data-Challenge-2026"):
    '''
    Takes a the set of tables from Data set and
    transforms into panda data frame for manipulation
    
    Input:
    -     table_type   = Set of tables of interest from Data set [str]
    -     data_path    = File Path which data is located in Local device, change as necessary[str]
    
    '''
    if table_type not in ['ball-positions', 'ball-events', 'player-positions', 'lineups', 'game-info']:
        print("Invalid data subset name. Please try again with a valid data subset.")
        return -1

    if table_type == 'lineups' or table_type == 'game-info':
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type + '.csv'), format = 'csv')
    else:
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type), format = 'csv', partitioning = ['home_team', 'away_team', 'year', 'day'])

In [4]:
#######################################################
### Code Taken from SMT_Data_Challenge_Demo
#######################################################

# Filter down to only 1 game for ease of computation
selected_game_str = "y1_d061_VKA_PHD"

# Read in the ball-events Data and filter to selected game
ball_events_subset = readDataSubset('ball-events')
ball_events = ball_events_subset.to_table(filter = (pads.field('game_string') == selected_game_str)).to_pandas()
ball_events_df = pl.from_pandas(ball_events)

# Read in the ball_pos Data and filter to selected game
ball_pos_subset = readDataSubset('ball-positions')
ball_pos = ball_pos_subset.to_table(filter = (pads.field('game_string') == selected_game_str)).to_pandas()
ball_pos_df = pl.from_pandas(ball_pos)

In [5]:
#get pickoff plays, add game string ppg
pickoff_plays_df = (
    ball_events_df.filter(pl.col("ball_eventcode") == 6)
    .with_columns(pl.concat_str(
        pl.col('game_string'),
        pl.col('play_per_game'),
        separator = "_"
    ).alias('game_string_ppg')
    ).select(
        ['game_string', 'play_per_game', 'game_string_ppg',
         pl.col('timestamp').alias('start_timestamp')]
    )
)

ball_pos_pickoffs = (
    ball_pos_df.with_columns(
        pl.concat_str(
            pl.col('game_string'),
            pl.col('play_per_game'),
            separator = '_'
        ).alias('game_string_ppg')
    )
).filter(
    pl.col('game_string_ppg').is_in(pickoff_plays_df['game_string_ppg'].to_list())
).join(
    pickoff_plays_df,
    how = "left",
    on = ['game_string', 'play_per_game', 'game_string_ppg']
)

ball_pos_pickoffs.head()

game_string,play_per_game,timestamp,ball_position_x,ball_position_y,ball_position_z,home_team,away_team,year,day,game_string_ppg,start_timestamp
str,i64,i64,f64,f64,f64,str,str,str,str,str,i64
"""y1_d061_VKA_PHD""",129,4140464,1.957431,61.4472,5.75184,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_129""",4140464
"""y1_d061_VKA_PHD""",129,4140514,6.86442,61.4607,5.82108,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_129""",4140464
"""y1_d061_VKA_PHD""",129,4140564,11.73021,61.4922,5.82423,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_129""",4140464
"""y1_d061_VKA_PHD""",129,4140614,16.55487,61.5417,5.7612,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_129""",4140464
"""y1_d061_VKA_PHD""",129,4140664,21.3384,61.6092,5.63208,"""PHD""","""VKA""","""y1""","""d061""","""y1_d061_VKA_PHD_129""",4140464
